# Aula 02 - Dados Financeiros: Do Preço ao Retorno

Módulo 08 AdmTech, Computação · Semana 4 · Sprint 2 · Prof. José Romualdo

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/josercf/2026-2B-M08/blob/main/notebooks/aula02.ipynb)

## O que este notebook é

Laboratório executável da Aula 02, que alimenta o artefato A2 (Módulo de Análise de Risco e Retorno Individual). Parte do Adj Close que o coletor da Aula 01 deixou pronto e chega à tabela de retorno e volatilidade anualizados da cesta, com o CDI do período como referência. Roda no Colab pelo link acima ou localmente a partir do repositório clonado. Sem rede, cada leitura usa o snapshot versionado em `dados/`; com `M08_OFFLINE=1`, a queda para a internet fica desligada.

## 1. O preço herdado da Aula 01

O retorno calculado hoje herda a decisão de ausências da Sprint 1. Este notebook usa a do exemplo de ADR da Aula 01: **preencher para a frente, ativo por ativo**, sem inventar preço antes da primeira cotação. Por isso a GOLD11 só entra a partir de 11/12/2020 e o BOVA11, o IVVB11 e a GOLD11 ganham retorno zero em 24/10/2025. O desafio da seção 8 inverte essa decisão e mede o efeito.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DADOS_LOCAL = os.path.join("..", "dados")
DADOS_BRUTOS = "https://raw.githubusercontent.com/josercf/2026-2B-M08/main/dados/"
PREGOES_ANO = 252


def ler_csv(nome, **kwargs):
    """Lê um CSV de dados/: do disco quando o repositório está clonado, da URL bruta no Colab.

    Com M08_OFFLINE=1 a queda para a rede fica desligada, como no CI.
    """
    local = os.path.join(DADOS_LOCAL, nome)
    if os.path.exists(local):
        return pd.read_csv(local, **kwargs)
    if os.environ.get("M08_OFFLINE") == "1":
        raise FileNotFoundError("%s não está em %s e M08_OFFLINE=1 desliga a rede" % (nome, DADOS_LOCAL))
    return pd.read_csv(DADOS_BRUTOS + nome, **kwargs)


ajustado = ler_csv("precos_ajustados.csv", index_col="data", parse_dates=["data"])
precos = ajustado.ffill()  # decisão do ADR de ausências da Aula 01
print("pregões:", len(precos), "de", precos.index[0].date(), "a", precos.index[-1].date())
print("ativos:", ", ".join(precos.columns))

## 2. Retorno simples e retorno logarítmico

O retorno simples de um pregão é `P_t / P_(t-1) - 1`, e é o que o `pct_change` calcula. O retorno logarítmico é `ln(P_t / P_(t-1))`, que é o mesmo que `ln(1 + r)`. Os dois quase coincidem em dias calmos e se afastam em dias de queda ou alta forte.

O argumento `fill_method=None` impede o pandas de preencher ausências por conta própria antes de calcular: a decisão de ausências é do grupo, registrada em ADR, e não um padrão escondido da biblioteca.

In [ ]:
simples = precos.pct_change(fill_method=None).iloc[1:]
log = np.log(precos / precos.shift(1)).iloc[1:]

dia = "2020-03-09"
print("PETR4 em 09/03/2020: simples %.2f%%, log %.2f%%"
      % (100 * simples.loc[dia, "PETR4"], 100 * log.loc[dia, "PETR4"]))

### A soma que erra de propósito

Na crise de 2020, de 20/02 a 30/04, a PETR4 teve 47 pregões. A célula abaixo soma os retornos simples desses pregões e compara com o acumulado de verdade, que compõe os retornos multiplicando `(1 + r)`. Antes de rodar, a dupla escreve qual das duas somas acerta o acumulado.

In [ ]:
crise_s = simples.loc["2020-02-20":"2020-04-30", "PETR4"]
crise_l = log.loc["2020-02-20":"2020-04-30", "PETR4"]
acumulado = (1 + crise_s).prod() - 1
print("pregões:", len(crise_s))
print("soma dos simples:   %.2f%%" % (100 * crise_s.sum()))
print("acumulado de fato:  %.2f%%" % (100 * acumulado))
print("soma dos log:       %.2f%%" % (100 * crise_l.sum()))
print("ln(1 + acumulado):  %.2f%%" % (100 * np.log(1 + acumulado)))

## 3. Média e desvio padrão diários

Prática em dupla do Bloco 3. A média diária é o retorno típico de um pregão; o desvio padrão diário mede o quanto os pregões se afastam dessa média, e é a medida de risco que o resto do módulo usa.

In [ ]:
diarias = pd.DataFrame({
    "média diária (%)": 100 * simples.mean(),
    "desvio diário (%)": 100 * simples.std(),
    "pregões": simples.count(),
}).sort_values("desvio diário (%)", ascending=False)
diarias.round(3)

### Pergunta para a dupla

1. Qual ativo tem o maior desvio diário, e qual tem o menor?
2. Por que a GOLD11 tem menos pregões que os demais?
3. A média diária da MGLU3 é negativa. O que isso diz sobre quem comprou no início da janela?

<details>
<summary>Resposta</summary>

1. MGLU3 tem o maior desvio diário; GOLD11, o menor.
2. O fundo só tem cotação a partir de 11/12/2020, e o preenchimento para a frente não inventa preço antes da primeira cotação.
3. Na média, a MGLU3 perdeu valor a cada pregão: quem comprou em 2019 e segurou até 31/08/2026 perdeu dinheiro, como a seção 6 confirma.

</details>

## 4. Histograma e caudas

O desvio padrão resume a dispersão em um número, e a curva normal com a mesma média e o mesmo desvio diz quantos pregões extremos esse número permitiria esperar. A célula conta os pregões além de 3 desvios e compara com o que a normal preveria.

In [ ]:
from math import erfc, sqrt

r = simples["PETR4"].dropna()
z = (r - r.mean()) / r.std()
alem = z[z.abs() > 3]
print("pregões além de 3 desvios: %d (a normal preveria %.1f)" % (len(alem), erfc(3 / sqrt(2)) * len(r)))
print("deles, em março de 2020:", int((alem.index.strftime("%Y-%m") == "2020-03").sum()))
print("pior queda: %.2f%% em %s, a %.2f desvios da média" % (100 * r.min(), r.idxmin().date(), z.min()))

fig, eixo = plt.subplots(figsize=(11, 4))
eixo.hist(r, bins=100)
eixo.axvline(r.mean(), linestyle="--", label="média")
eixo.axvline(r.mean() - r.std(), linestyle=":", label="média menos um desvio")
eixo.axvline(r.mean() + r.std(), linestyle=":", label="média mais um desvio")
eixo.set_title("PETR4: retornos diários simples")
eixo.legend()
plt.show()

## 5. Anualização

O retorno médio anual multiplica a média diária por 252 pregões. A volatilidade anual multiplica o desvio diário pela **raiz** de 252, porque o que se soma ao longo de pregões independentes é a variância, e o desvio é a raiz da variância. Multiplicar o desvio por 252 é o erro mais comum do A2.

In [ ]:
media_d = simples["PETR4"].mean()
desvio_d = simples["PETR4"].std()
print("média diária %.4f%% x 252 = %.2f%% ao ano" % (100 * media_d, 100 * media_d * PREGOES_ANO))
print("desvio diário %.4f%% x raiz de 252 (%.2f) = %.2f%% ao ano"
      % (100 * desvio_d, np.sqrt(PREGOES_ANO), 100 * desvio_d * np.sqrt(PREGOES_ANO)))
print("erro comum, desvio x 252 = %.2f%% ao ano" % (100 * desvio_d * PREGOES_ANO))

### Volatilidade em janela deslizante

A volatilidade da janela inteira é um número só para sete anos e meio. A janela deslizante recalcula o desvio a cada pregão, olhando para trás 21 pregões (um mês), 63 (um trimestre) ou 252 (um ano). Antes de rodar, a dupla aposta qual curva atinge o pico primeiro em 2020.

In [ ]:
fig, eixo = plt.subplots(figsize=(11, 4))
for janela in (21, 63, 252):
    movel = simples["BOVA11"].rolling(janela).std() * np.sqrt(PREGOES_ANO)
    pico = movel.loc["2020"]
    print("%3d pregões: pico de %.2f%% em %s" % (janela, 100 * pico.max(), pico.idxmax().date()))
    eixo.plot(100 * movel, label="%d pregões" % janela)
eixo.axhline(100 * simples["BOVA11"].std() * np.sqrt(PREGOES_ANO), linestyle="--", label="janela inteira")
eixo.set_ylabel("volatilidade anualizada (%)")
eixo.set_title("BOVA11: volatilidade em janela deslizante")
eixo.legend()
plt.show()

## 6. A cesta inteira contra o CDI

Prática em dupla do Bloco 6. A tabela reúne o retorno médio anual, a volatilidade anual e o retorno total de cada ativo, da primeira à última cotação. O CDI de `cdi_diario.csv` vem em porcentagem ao dia e se acumula pelo produto de `(1 + taxa)`; cada ativo é comparado com o CDI acumulado desde a primeira cotação dele. O retorno anual da tabela é a média diária vezes 252, uma média aritmética: a comparação com o CDI usa o retorno total contra o CDI acumulado no mesmo período, nunca a média anual contra a taxa anual composta do CDI.

In [ ]:
cdi = ler_csv("cdi_diario.csv", index_col="data", parse_dates=["data"])["valor"] / 100


def cdi_acumulado(inicio):
    return (1 + cdi.loc[inicio:]).prod() - 1


inicio = precos.apply(lambda s: s.first_valid_index())
total = precos.apply(lambda s: s.dropna().iloc[-1] / s.dropna().iloc[0] - 1)
tabela = pd.DataFrame({
    "retorno anual (%)": 100 * simples.mean() * PREGOES_ANO,
    "volatilidade anual (%)": 100 * simples.std() * np.sqrt(PREGOES_ANO),
    "retorno total (%)": 100 * total,
    "CDI no mesmo período (%)": [100 * cdi_acumulado(inicio[t]) for t in precos.columns],
}).sort_values("volatilidade anual (%)", ascending=False)
tabela["acima do CDI"] = tabela["retorno total (%)"] > tabela["CDI no mesmo período (%)"]
print("CDI acumulado na janela inteira: %.2f%%" % (100 * cdi_acumulado(None)))
tabela.round(2)

### Pergunta para a dupla

Julguem cada afirmação como sustentada ou não pelos números da tabela, antes de abrir a resposta.

1. A MGLU3 é o ativo de maior risco da cesta.
2. Na cesta, quem oscila mais rende mais.
3. O BOVA11 rendeu mais que o CDI na janela.
4. A PETR4 entregou o maior retorno total da cesta.

<details>
<summary>Resposta</summary>

1. Sustentada: a maior volatilidade anual da tabela.
2. Não sustentada: a MGLU3 tem a maior volatilidade e o pior retorno total.
3. Não sustentada: o retorno total do BOVA11 fica abaixo do CDI acumulado no mesmo período.
4. Sustentada: o maior retorno total da coluna.

</details>

## 7. O mínimo de covariância e correlação para o A2

O A2 pede matriz de correlação com heatmap e matriz de covariância anualizada, e a Aula 03 só ensina esse conteúdo na semana 5 (pendência P5 do plano de ensino). As duas linhas abaixo são o mínimo para a entrega da Sprint 2: a covariância anualiza por 252, como a variância, e a correlação não anualiza, porque é adimensional. A leitura de quais pares andam juntos é assunto da Aula 03.

In [ ]:
covariancia_anual = simples.cov() * PREGOES_ANO
correlacao = simples.corr()

fig, eixo = plt.subplots(figsize=(8, 7))
imagem = eixo.imshow(correlacao, vmin=-1, vmax=1, cmap="RdBu_r")
eixo.set_xticks(range(len(correlacao)), correlacao.columns, rotation=90)
eixo.set_yticks(range(len(correlacao)), correlacao.columns)
fig.colorbar(imagem, ax=eixo, label="correlação")
eixo.set_title("Correlação dos retornos diários")
plt.show()
print("a diagonal da covariância anual é a variância anual: PETR4 %.4f = %.4f ao quadrado"
      % (covariancia_anual.loc["PETR4", "PETR4"], np.sqrt(covariancia_anual.loc["PETR4", "PETR4"])))

## 8. Desafio para o A2

a. Repetir a tabela da seção 6 com a decisão de ausências invertida: `ajustado.dropna()` na cesta inteira no lugar de `ajustado.ffill()`.
b. Medir quanto a volatilidade anual de cada ativo muda entre as duas decisões e apontar o ativo que mais muda.
c. Registrar o resultado no ADR da anualização do grupo, com o número antes e depois.
d. Transformar as seções 2 a 6 em funções que recebem o DataFrame do A1 e devolvem a tabela, e escrever no `SPEC.md` um caso de teste com os dados do grupo.

In [ ]:
# Ponto de partida do desafio: complete com a comparação pedida.
descartar = ajustado.dropna()
print("pregões com dropna:", len(descartar), "de", len(ajustado))